# Routers and URL Versioning

This notebook covers:

1. Split a large app into multiple `APIRouter`s
2. Mount routers under URL prefixes (`/v1`, `/v2`)
3. Apply per-router tags, dependencies, and response models
4. Run v1 and v2 side-by-side with shared code

**Scope**: FastAPI + `TestClient`.

By the end, the structure here is exactly how the capstone organizes `/assets`, `/portfolios`, `/holdings`, `/pricing` — and how it would absorb a `/v2`.

## 1. The Single-File Wall

A FastAPI app starts small. Two endpoints fit comfortably in `main.py`. Ten still work. By thirty, you've hit the wall: route declarations scroll forever, related endpoints live nowhere near each other, and changing one resource's auth means scanning the whole file.

`APIRouter` is the answer. A router is a collection of routes that gets *mounted* into the app under a prefix. Each resource gets its own router file; `main.py` only wires them together.

A realistic portfolio app:

```
app/
├── main.py                  # FastAPI(), include each router
└── routers/
    ├── assets.py            # APIRouter(prefix="/assets", tags=["assets"])
    ├── portfolios.py        # APIRouter(prefix="/portfolios", ...)
    ├── holdings.py
    └── pricing.py
```

The notebook examples below stay inline (no extra files), but the pattern is the one above.

## 2. `APIRouter` Basics

`APIRouter` has the same `.get`, `.post`, etc. decorators as `FastAPI`. You build it standalone, then `app.include_router(it)` to attach.

A router on its own is inert — nothing routes to it until it's included.

In [ ]:
from fastapi import APIRouter, FastAPI
from fastapi.testclient import TestClient

# Build a router. No app yet — the router doesn't care.
assets_router = APIRouter()

@assets_router.get("/assets")
def list_assets():
    return [{"ticker": "AAPL"}, {"ticker": "MSFT"}]

@assets_router.get("/assets/{ticker}")
def get_asset(ticker: str):
    return {"ticker": ticker}

# Now wire it into an app.
app = FastAPI()
app.include_router(assets_router)

client = TestClient(app)
print("GET /assets       ->", client.get("/assets").json())
print("GET /assets/AAPL  ->", client.get("/assets/AAPL").json())

## 3. Mounting Routers with Prefixes

`include_router(..., prefix="/foo")` prepends `/foo` to every route the router declares. So a router can stay agnostic about where it's mounted — the same router can be served under `/assets` here and `/internal/assets` in an admin app.

Combined with the constructor form `APIRouter(prefix="/assets")`, you have two equally valid styles:

- Prefix on the router (declared once, mounted bare): `APIRouter(prefix="/assets")` + `include_router(r)`
- Prefix at inclusion (router declared bare, mounted with prefix): `APIRouter()` + `include_router(r, prefix="/assets")`

Pick one and stay consistent. The capstone uses the constructor form.

In [ ]:
# Style A: prefix on the router itself
assets_router = APIRouter(prefix="/assets", tags=["assets"])

@assets_router.get("")          # final URL: /assets
def list_assets():
    return [{"ticker": "AAPL"}]

@assets_router.get("/{ticker}")  # final URL: /assets/{ticker}
def get_asset(ticker: str):
    return {"ticker": ticker}

# Style B: bare router, prefix when including
portfolios_router = APIRouter()

@portfolios_router.get("")
def list_portfolios():
    return [{"id": 1, "name": "Growth"}]

app = FastAPI()
app.include_router(assets_router)
app.include_router(portfolios_router, prefix="/portfolios", tags=["portfolios"])

client = TestClient(app)
print("GET /assets       ->", client.get("/assets").json())
print("GET /assets/AAPL  ->", client.get("/assets/AAPL").json())
print("GET /portfolios   ->", client.get("/portfolios").json())

print("\nregistered routes:")
for r in app.routes:
    if hasattr(r, "methods") and r.path not in {"/openapi.json", "/docs", "/redoc", "/docs/oauth2-redirect"}:
        print(f"  {','.join(r.methods):10} {r.path}")

## 4. URL Versioning: `/v1`, `/v2`

Once an API has external clients, breaking changes need a version namespace. The cleanest pattern is **URL versioning** — the version is part of the path: `/v1/assets`, `/v2/assets`.

The mechanics are just nested prefixes: include each resource router into a *version* router, then include the version router into the app.

```
app
└── /v1
    ├── /v1/assets       ← assets_router under v1
    └── /v1/portfolios   ← portfolios_router under v1
└── /v2
    ├── /v2/assets       ← assets_router_v2 (new shape)
    └── /v2/portfolios   ← portfolios_router (shared, no breaking change)
```

Versions co-exist until clients migrate. When v1 is sunset, deprecate, then delete the v1 wiring — the routers themselves can stay if v2 still uses them.

In [ ]:
# v1 routers
v1_assets = APIRouter(prefix="/assets", tags=["assets"])

@v1_assets.get("/{ticker}")
def get_asset_v1(ticker: str):
    # v1 shape: flat
    return {"ticker": ticker, "price": 195.0}

# v2 routers — same resource, new shape (price wrapped with currency)
v2_assets = APIRouter(prefix="/assets", tags=["assets"])

@v2_assets.get("/{ticker}")
def get_asset_v2(ticker: str):
    return {"ticker": ticker, "price": {"amount": 195.0, "currency": "USD"}}

# Compose under version routers
v1 = APIRouter(prefix="/v1")
v1.include_router(v1_assets)

v2 = APIRouter(prefix="/v2")
v2.include_router(v2_assets)

app = FastAPI()
app.include_router(v1)
app.include_router(v2)

client = TestClient(app)
print("GET /v1/assets/AAPL ->", client.get("/v1/assets/AAPL").json())
print("GET /v2/assets/AAPL ->", client.get("/v2/assets/AAPL").json())

## 5. Tags, Dependencies, and Defaults at the Router Level

A router can apply defaults to every route it owns:

- **`tags=[...]`** — every route gets these tags in the docs.
- **`dependencies=[...]`** — every route runs these deps (auth checks, rate limit gates, request-context). Full DI coverage is in chapter 4.
- **`responses={...}`** — every route advertises these alternate response schemas in OpenAPI (e.g., a shared 401 shape).

Setting defaults at the router level means you can't forget them on a new route. New endpoint added to the assets router? It's already protected.

In [ ]:
from fastapi import Depends, Header, HTTPException

# Pretend-auth dep (real version in chapter 5). Every route in this router runs it.
# Use Header(None) — missing-header and wrong-value both return a uniform 401,
# which is the production-friendly pattern (don't leak whether the header was sent).
def require_api_key(x_api_key: str | None = Header(default=None)):
    if x_api_key != "secret":
        raise HTTPException(status_code=401, detail="bad or missing api key")
    return x_api_key

protected_assets = APIRouter(
    prefix="/assets",
    tags=["assets"],
    dependencies=[Depends(require_api_key)],
    responses={401: {"description": "Bad or missing API key"}},
)

@protected_assets.get("/{ticker}")
def get_asset(ticker: str):
    return {"ticker": ticker}

app = FastAPI()
app.include_router(protected_assets)

client = TestClient(app)
print("no key   ->", client.get("/assets/AAPL").status_code, client.get("/assets/AAPL").json())
print("bad key  ->", client.get("/assets/AAPL", headers={"x-api-key": "wrong"}).status_code)
print("good key ->", client.get("/assets/AAPL", headers={"x-api-key": "secret"}).json())

# OpenAPI advertises the 401 response shape on every route in this router.
schema = client.get("/openapi.json", headers={"x-api-key": "secret"}).json()
print("\nresponses declared for /assets/{ticker}:")
print(" ", list(schema["paths"]["/assets/{ticker}"]["get"]["responses"]))

## 6. Sharing Logic Between v1 and v2

The point of versioning is to let the API shape evolve without breaking clients. The point of routers is to share *implementation* between versions whenever the underlying behavior hasn't changed.

A common pattern:

- **Service / repository layer** holds the actual logic (fetch the asset).
- **v1 and v2 routers** each serialize the service's output into their version's shape.
- **Bug fixes and storage changes** happen once in the service. Both versions inherit them.
- **Truly different behavior** (e.g., v2 adds a new field that v1 can't represent) lives in the v2 router only.

The skeleton below shows the split. Notice the service is shared; only the response shape differs.

In [ ]:
# Shared "service" — the real logic, version-agnostic.
def fetch_asset(ticker: str) -> dict:
    return {"ticker": ticker.upper(), "price_usd": 195.0, "name": "Apple Inc."}

# v1 serializer: flat price
v1_router = APIRouter(prefix="/v1/assets", tags=["assets-v1"])

@v1_router.get("/{ticker}")
def v1_get(ticker: str):
    a = fetch_asset(ticker)
    return {"ticker": a["ticker"], "price": a["price_usd"]}

# v2 serializer: structured price object + name
v2_router = APIRouter(prefix="/v2/assets", tags=["assets-v2"])

@v2_router.get("/{ticker}")
def v2_get(ticker: str):
    a = fetch_asset(ticker)
    return {
        "ticker": a["ticker"],
        "name": a["name"],
        "price": {"amount": a["price_usd"], "currency": "USD"},
    }

app = FastAPI()
app.include_router(v1_router)
app.include_router(v2_router)

client = TestClient(app)
print("v1 ->", client.get("/v1/assets/aapl").json())
print("v2 ->", client.get("/v2/assets/aapl").json())
print("\nfix `fetch_asset` once -> both versions benefit. That's the whole point.")

## Key Takeaways

- **One `APIRouter` per resource.** `assets.py`, `portfolios.py`, `holdings.py`. `main.py` only wires.
- **Mount with a prefix.** `app.include_router(r, prefix="/v1")` or `APIRouter(prefix="/v1")` — pick one style, be consistent.
- **Router-level defaults** (`tags`, `dependencies`, `responses`) apply to every route on the router. New routes inherit them automatically — that's how shared auth stays shared.
- **URL versioning is nested includes.** A `/v1` router includes per-resource routers. Versions co-exist; sunset by removing the version router's `include_router` line.
- **Share the service layer.** Routers serialize; services do the work. Bug fixes and storage changes happen once.

Chapter 2 done. Next chapter swaps lanes from request handling to performance — async vs sync, blocking the event loop, background tasks and streaming.

## Exercises

**1. Split a monolithic app.** Take an inline app that defines both `/assets/*` and `/portfolios/*` endpoints in one file. Refactor into two routers (`assets_router`, `portfolios_router`), each with its own prefix and tag. Confirm `/openapi.json` groups them correctly.

**2. Add a `/v2`.** Take the v1 assets router (flat `price`) and add a v2 router (structured `price: {amount, currency}`). Both should be reachable simultaneously. Use a shared `fetch_asset` function so a bug fix in the service is picked up by both versions.

**3. Shared auth.** Build a router with `dependencies=[Depends(require_api_key)]` and a route that doesn't reference the dependency anywhere in its signature. Confirm the dep still runs (TestClient should return 401 without the header).